In [2]:
###Transformer训练AML风险化学物筛选

In [2]:
!pip install pandas numpy torch scikit-learn optuna shap matplotlib

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 26.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 26.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.7/591.7 kB 53.6 MB/s eta 0:00:00


In [1]:
# ===================== 0. 导入库与设置随机种子 =====================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import optuna
from optuna.trial import TrialState
import os
import warnings
import matplotlib.pyplot as plt
import random
import shap
import json
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

warnings.filterwarnings('ignore')

# 随机种子
SEED = 3407
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"✅ 随机种子已设置为: {seed}")

# 全局配置
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE_BASE = 32          
MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8
MAX_SEQ_LEN = 64
FEAT_DIM = 119
MODEL_SAVE_DIR = "chem_models_Transformer"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)


/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===================== 1. 定义模型与数据集类 =====================
class SimpleTransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=2, num_layers=1, dropout=0.1):
        super().__init__()
        self.smiles_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(MAX_SEQ_LEN, d_model)
        self.feat_proj = nn.Linear(FEAT_DIM, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2,
            dropout=dropout, activation="gelu", batch_first=True, device=DEVICE
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(d_model*2, d_model), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_model, 1)
        )
    def forward(self, smiles_seq, feat):
        batch_size = smiles_seq.shape[0]
        pos = torch.arange(0, MAX_SEQ_LEN).unsqueeze(0).repeat(batch_size, 1).to(DEVICE)
        smiles_feat = self.smiles_emb(smiles_seq) + self.pos_emb(pos)
        padding_mask = (smiles_seq == 0)
        enc_out = self.transformer_encoder(smiles_feat, src_key_padding_mask=padding_mask)
        smiles_pooled = enc_out.masked_fill(padding_mask.unsqueeze(-1), 0).sum(1)
        valid_tokens = (~padding_mask).sum(1).unsqueeze(-1)
        valid_tokens = torch.where(valid_tokens == 0, torch.ones_like(valid_tokens), valid_tokens)  # 避免除零
        smiles_pooled = smiles_pooled / valid_tokens 
        feat_proj = self.feat_proj(feat)
        concat_feat = torch.cat([smiles_pooled, feat_proj], dim=1)
        out = self.classifier(concat_feat)
        return out

class ChemDataset(Dataset):
    def __init__(self, smiles, features, labels, smiles_vocab, scaler=None):
        self.smiles = smiles
        self.features = features
        self.labels = labels
        self.smiles_vocab = smiles_vocab
        self.scaler = scaler
        if self.scaler is not None:
            self.features = self.scaler.transform(self.features)
    def _smiles_to_seq(self, smiles):
        seq = [self.smiles_vocab.get(c, 1) for c in smiles[:MAX_SEQ_LEN]]
        if len(seq) < MAX_SEQ_LEN:
            seq += [0] * (MAX_SEQ_LEN - len(seq))
        return np.array(seq)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        smiles_seq = torch.LongTensor(self._smiles_to_seq(self.smiles[idx]))
        feat = torch.FloatTensor(self.features[idx])
        label = torch.FloatTensor([self.labels[idx]])
        return smiles_seq, feat, label

def calculate_metrics(y_true, y_pred_prob, threshold=0.5):
    y_pred = (y_pred_prob >= threshold).astype(int)
    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    except:
        tn = fp = fn = tp = 0
        if (y_true == 0).all():
            tn = len(y_true)
        elif (y_true == 1).all():
            tp = len(y_true)
    total = tp + tn + fp + fn
    metrics = {
        "accuracy": (tp + tn) / total if total > 0 else 0.0,
        "sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "precision": tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        "f1": 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0,
        "auc": roc_auc_score(y_true, y_pred_prob) if len(set(y_true)) > 1 else 0.5
    }
    return metrics


In [3]:
# ===================== 2. 数据加载与预处理 =====================
def load_and_preprocess(csv_path, target_col="Activity"):
    """读取CSV，二值化标签，返回DataFrame及特征列名"""
    df = pd.read_csv(csv_path)
    smiles_col = "SMILES" if "SMILES" in df.columns else "smiles"
    # 标签二值化
    df[target_col] = df[target_col].str.strip().str.lower().map({"active": 1, "inactive": 0})
    df = df.dropna(subset=[target_col])
    df[target_col] = df[target_col].astype(int)
    # 特征列
    feat_cols = [col for col in df.columns if col not in [smiles_col, target_col]]
    if len(feat_cols) != FEAT_DIM:
        raise ValueError(f"分子特征数量不为{FEAT_DIM}，当前为{len(feat_cols)}")
    return df, smiles_col, feat_cols, target_col

def build_vocab(df, smiles_col, min_freq=5):
    smiles_list = df[smiles_col].tolist()
    char_counts = {}
    for s in smiles_list:
        for c in s:
            char_counts[c] = char_counts.get(c, 0) + 1
    vocab = {c: i+2 for i, (c, cnt) in enumerate(char_counts.items()) if cnt > min_freq}
    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1
    return vocab

In [4]:
# ===================== 3. Optuna 目标函数（内层） =====================
def objective(trial, train_loader, val_loader, vocab, pos_weight):
    """内层优化目标，返回最佳验证集AUC及损失曲线"""
    set_seed(SEED)  # 保证每次试验可重现
    params = {
        "d_model": trial.suggest_categorical("d_model", [32, 64]),
        "nhead": trial.suggest_categorical("nhead", [2, 4]),
        "num_layers": trial.suggest_int("num_layers", 1, 2),
        "dropout": trial.suggest_float("dropout", 0.1, 0.2),
        "lr": trial.suggest_float("lr", 1e-4, 5e-4, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-4, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [8, 16])
    }
    model = SimpleTransformerModel(
        vocab_size=len(vocab),
        d_model=params["d_model"],
        nhead=params["nhead"],
        num_layers=params["num_layers"],
        dropout=params["dropout"]
    ).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))
    optimizer = optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    
    best_val_auc = 0.0
    patience_counter = 0
    train_losses, val_losses = [], []
    
    for epoch in range(MAX_EPOCHS):
        # 训练
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            smiles_seq, feat, label = [x.to(DEVICE) for x in batch]
            optimizer.zero_grad()
            logits = model(smiles_seq, feat)
            loss = criterion(logits, label)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        # 验证
        model.eval()
        val_loss = 0.0
        val_preds_prob, val_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                smiles_seq, feat, label = [x.to(DEVICE) for x in batch]
                logits = model(smiles_seq, feat)
                loss = criterion(logits, label)
                val_loss += loss.item()
                pred_prob = torch.sigmoid(logits).cpu().numpy().flatten()
                val_preds_prob.extend(pred_prob)
                val_labels.extend(label.cpu().numpy().flatten())
        val_loss /= len(val_loader)
        val_auc = roc_auc_score(val_labels, val_preds_prob) if len(set(val_labels))>1 else 0.5
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                break
    return best_val_auc, train_losses, val_losses


In [7]:
# ===================== 4. 嵌套交叉验证主流程 =====================
def nested_cv(df, smiles_col, feat_cols, target_col, vocab, outer_k=5, inner_trials=6):
    """执行嵌套交叉验证，返回各折指标、学习曲线损失及最佳参数列表"""
    X_all_feat = df[feat_cols].fillna(0).values
    y_all = df[target_col].values
    smiles_all = df[smiles_col].tolist()
    
    skf = StratifiedKFold(n_splits=outer_k, shuffle=True, random_state=SEED)
    
    outer_scores = []           # 每折测试集指标
    all_train_losses = []       # 每折训练损失列表
    all_val_losses = []         # 每折验证损失列表
    best_params_list = []       # 每折最优超参数
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X_all_feat, y_all)):
        print(f"\n========== Outer Fold {fold+1}/{outer_k} ==========")
        
        # 获取该折训练集和测试集
        train_feat = X_all_feat[train_idx]
        train_label = y_all[train_idx]
        train_smiles = [smiles_all[i] for i in train_idx]
        test_feat = X_all_feat[test_idx]
        test_label = y_all[test_idx]
        test_smiles = [smiles_all[i] for i in test_idx]
        
        # 计算该折的类别权重（仅使用训练集）
        pos_num = np.sum(train_label == 1)
        neg_num = np.sum(train_label == 0)
        pos_weight = neg_num / pos_num if pos_num > 0 else 1.0
        
        # 标准化：在训练集上拟合
        scaler = StandardScaler()
        train_feat_scaled = scaler.fit_transform(train_feat)
        test_feat_scaled = scaler.transform(test_feat)
        
        # ---------- 内层划分：从训练集中再划分验证集（用于Optuna） ----------
        X_train_inner, X_val_inner, y_train_inner, y_val_inner, smiles_train_inner, smiles_val_inner = train_test_split(
            train_feat_scaled, train_label, train_smiles, test_size=0.3, random_state=SEED, stratify=train_label
        )
        train_dataset_inner = ChemDataset(smiles_train_inner, X_train_inner, y_train_inner, vocab, scaler=None)
        val_dataset_inner = ChemDataset(smiles_val_inner, X_val_inner, y_val_inner, vocab, scaler=None)
        train_loader_inner = DataLoader(train_dataset_inner, batch_size=BATCH_SIZE_BASE, shuffle=True, num_workers=0)
        val_loader_inner = DataLoader(val_dataset_inner, batch_size=BATCH_SIZE_BASE*2, shuffle=False, num_workers=0)
        
        # ---------- Optuna 超参数优化 ----------
        def objective_wrapper(trial):
            best_auc, _, _ = objective(trial, train_loader_inner, val_loader_inner, vocab, pos_weight)
            return best_auc
        
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(objective_wrapper, n_trials=inner_trials, timeout=3600)
        best_params = study.best_params
        best_params_list.append(best_params)
        print(f"Best params: {best_params}")
        print(f"Best inner validation AUC: {study.best_value:.4f}")
        
        # ---------- 使用最优参数在整个外层训练集上重新训练模型 ----------
        train_dataset_full = ChemDataset(train_smiles, train_feat_scaled, train_label, vocab, scaler=None)
        train_loader_full = DataLoader(train_dataset_full, batch_size=best_params["batch_size"], shuffle=True, num_workers=0)
        test_dataset_full = ChemDataset(test_smiles, test_feat_scaled, test_label, vocab, scaler=None)
        test_loader_full = DataLoader(test_dataset_full, batch_size=BATCH_SIZE_BASE*2, shuffle=False, num_workers=0)
        
        model = SimpleTransformerModel(
            vocab_size=len(vocab),
            d_model=best_params["d_model"],
            nhead=best_params["nhead"],
            num_layers=best_params["num_layers"],
            dropout=best_params["dropout"]
        ).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))
        optimizer = optim.AdamW(model.parameters(), lr=best_params["lr"], weight_decay=best_params["weight_decay"])
        
        # 记录该折的训练/验证损失（使用内层验证集）
        train_losses_fold, val_losses_fold = [], []
        best_val_auc_full = 0.0
        patience_counter = 0
        best_model_state = None
        
        for epoch in range(MAX_EPOCHS):
            # 训练
            model.train()
            train_loss = 0.0
            for batch in train_loader_full:
                smiles_seq, feat, label = [x.to(DEVICE) for x in batch]
                optimizer.zero_grad()
                logits = model(smiles_seq, feat)
                loss = criterion(logits, label)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            train_loss /= len(train_loader_full)
            
            # 验证（使用内层验证集）
            model.eval()
            val_loss = 0.0
            val_preds_prob, val_labels = [], []
            with torch.no_grad():
                for batch in val_loader_inner:
                    smiles_seq, feat, label = [x.to(DEVICE) for x in batch]
                    logits = model(smiles_seq, feat)
                    loss = criterion(logits, label)
                    val_loss += loss.item()
                    pred_prob = torch.sigmoid(logits).cpu().numpy().flatten()
                    val_preds_prob.extend(pred_prob)
                    val_labels.extend(label.cpu().numpy().flatten())
            val_loss /= len(val_loader_inner)
            val_auc = roc_auc_score(val_labels, val_preds_prob) if len(set(val_labels))>1 else 0.5
            
            train_losses_fold.append(train_loss)
            val_losses_fold.append(val_loss)
            
            if val_auc > best_val_auc_full:
                best_val_auc_full = val_auc
                patience_counter = 0
                best_model_state = model.state_dict().copy()
            else:
                patience_counter += 1
                if patience_counter >= EARLY_STOPPING_PATIENCE:
                    break
        
        # 加载该折最佳模型
        model.load_state_dict(best_model_state)
        
        # 测试集评估
        model.eval()
        test_preds_prob, test_labels = [], []
        with torch.no_grad():
            for batch in test_loader_full:
                smiles_seq, feat, label = [x.to(DEVICE) for x in batch]
                logits = model(smiles_seq, feat)
                pred_prob = torch.sigmoid(logits).cpu().numpy().flatten()
                test_preds_prob.extend(pred_prob)
                test_labels.extend(label.cpu().numpy().flatten())
        
        metrics = calculate_metrics(np.array(test_labels), np.array(test_preds_prob))
        print(f"Fold {fold+1} test metrics: Acc={metrics['accuracy']:.4f}, AUC={metrics['auc']:.4f}, Sen={metrics['sensitivity']:.4f}, Spe={metrics['specificity']:.4f}, F1={metrics['f1']:.4f}")
        outer_scores.append(metrics)
        all_train_losses.append(train_losses_fold)
        all_val_losses.append(val_losses_fold)
    
    # 汇总结果
    aucs = [m['auc'] for m in outer_scores]
    accs = [m['accuracy'] for m in outer_scores]
    sens = [m['sensitivity'] for m in outer_scores]
    spec = [m['specificity'] for m in outer_scores]
    
    print("\n===== Nested Cross-Validation Results =====")
    print(f"AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    print(f"Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"Sensitivity: {np.mean(sens):.4f} ± {np.std(sens):.4f}")
    print(f"Specificity: {np.mean(spec):.4f} ± {np.std(spec):.4f}")
    
    return outer_scores, all_train_losses, all_val_losses, best_params_list

In [8]:
# ===================== 5. 学习曲线绘制 =====================
def plot_learning_curves(all_train_losses, all_val_losses, save_path):
    """绘制平均学习曲线（带标准差）"""
    min_epochs = min(len(losses) for losses in all_train_losses)
    train_aligned = [losses[:min_epochs] for losses in all_train_losses]
    val_aligned = [losses[:min_epochs] for losses in all_val_losses]
    
    mean_train = np.mean(train_aligned, axis=0)
    std_train = np.std(train_aligned, axis=0)
    mean_val = np.mean(val_aligned, axis=0)
    std_val = np.std(val_aligned, axis=0)
    
    epochs = range(1, min_epochs+1)
    plt.figure(figsize=(10,6))
    plt.plot(epochs, mean_train, 'b-', label='Training Loss')
    plt.fill_between(epochs, mean_train-std_train, mean_train+std_train, alpha=0.2, color='b')
    plt.plot(epochs, mean_val, 'r-', label='Validation Loss')
    plt.fill_between(epochs, mean_val-std_val, mean_val+std_val, alpha=0.2, color='r')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Learning Curves (Average over Outer Folds)')
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ 学习曲线保存至: {save_path}")


In [9]:
# ===================== 6. 最终模型训练（全数据） =====================
def train_final_model(df, smiles_col, feat_cols, target_col, vocab, best_params, save_dir):
    """使用最优超参数在全部数据上训练最终模型并保存"""
    X_all_feat = df[feat_cols].fillna(0).values
    y_all = df[target_col].values
    smiles_all = df[smiles_col].tolist()
    
    # 计算全局类别权重（可选）
    pos_num = np.sum(y_all == 1)
    neg_num = np.sum(y_all == 0)
    pos_weight = neg_num / pos_num if pos_num > 0 else 1.0
    
    # 标准化
    scaler = StandardScaler()
    X_all_feat_scaled = scaler.fit_transform(X_all_feat)
    
    # 构建数据集（全数据）
    full_dataset = ChemDataset(smiles_all, X_all_feat_scaled, y_all, vocab, scaler=None)
    full_loader = DataLoader(full_dataset, batch_size=best_params["batch_size"], shuffle=True, num_workers=0)
    
    # 再划分一个验证集用于早停（从全数据中取20%）
    X_train_final, X_val_final, y_train_final, y_val_final, smiles_train_final, smiles_val_final = train_test_split(
        X_all_feat_scaled, y_all, smiles_all, test_size=0.2, random_state=SEED, stratify=y_all
    )
    train_dataset_final = ChemDataset(smiles_train_final, X_train_final, y_train_final, vocab, scaler=None)
    val_dataset_final = ChemDataset(smiles_val_final, X_val_final, y_val_final, vocab, scaler=None)
    train_loader_final = DataLoader(train_dataset_final, batch_size=best_params["batch_size"], shuffle=True, num_workers=0)
    val_loader_final = DataLoader(val_dataset_final, batch_size=BATCH_SIZE_BASE*2, shuffle=False, num_workers=0)
    
    # 模型
    model = SimpleTransformerModel(
        vocab_size=len(vocab),
        d_model=best_params["d_model"],
        nhead=best_params["nhead"],
        num_layers=best_params["num_layers"],
        dropout=best_params["dropout"]
    ).to(DEVICE)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))
    optimizer = optim.AdamW(model.parameters(), lr=best_params["lr"], weight_decay=best_params["weight_decay"])
    
    best_val_auc = 0.0
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(MAX_EPOCHS):
        # 训练
        model.train()
        train_loss = 0.0
        for batch in train_loader_final:
            smiles_seq, feat, label = [x.to(DEVICE) for x in batch]
            optimizer.zero_grad()
            logits = model(smiles_seq, feat)
            loss = criterion(logits, label)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader_final)
        
        # 验证
        model.eval()
        val_preds_prob, val_labels = [], []
        with torch.no_grad():
            for batch in val_loader_final:
                smiles_seq, feat, label = [x.to(DEVICE) for x in batch]
                logits = model(smiles_seq, feat)
                pred_prob = torch.sigmoid(logits).cpu().numpy().flatten()
                val_preds_prob.extend(pred_prob)
                val_labels.extend(label.cpu().numpy().flatten())
        val_auc = roc_auc_score(val_labels, val_preds_prob) if len(set(val_labels))>1 else 0.5
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                break
    
    # 加载最佳模型
    model.load_state_dict(best_model_state)
    
    # 保存模型及元数据
    save_path = os.path.join(save_dir, "/root/final_transformer_model.pth")
    torch.save({
        "model_state_dict": model.state_dict(),
        "best_params": best_params,
        "vocab": vocab,
        "feat_cols": feat_cols,
        "scaler": scaler,
        "model_config": {
            "vocab_size": len(vocab),
            "d_model": best_params["d_model"],
            "nhead": best_params["nhead"],
            "num_layers": best_params["num_layers"],
            "dropout": best_params["dropout"],
            "max_seq_len": MAX_SEQ_LEN,
            "feat_dim": FEAT_DIM
        },
        "training_info": {
            "pos_weight": pos_weight,
            "best_val_auc": best_val_auc,
            "device": str(DEVICE),
            "save_timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
        }
    }, save_path)
    print(f"✅ 最终模型已保存至: {save_path}")
    return model, scaler, val_loader_final

In [10]:
# ===================== SHAP分析（GPU优化版） =====================
def shap_analysis(model, val_loader, scaler, feat_cols, device='cuda'):
    """
    针对分子特征的SHAP分析（GPU加速版）
    参数：
        model: 已训练好的PyTorch模型（应在GPU上）
        val_loader: 验证集DataLoader
        scaler: 特征标准化器（此处未使用，仅保持接口一致）
        feat_cols: 特征列名列表
        device: 计算设备（'cuda'或'cpu'）
    """
    # 确保模型在指定设备
    model.to(device)
    model.eval()

    # 在提取 val_feat 之前检查
    for i, (smiles_seq, feat, label) in enumerate(val_loader):
        if (smiles_seq == 0).all(dim=1).any():
            print(f"警告: 验证集中存在全零序列，索引 {i}")
    
    # 1. 提取验证集分子特征和参考SMILES序列
    val_feat_list = []
    val_label_list = []
    ref_smiles_seq = None
   
    
    with torch.no_grad():
        for batch in val_loader:
            smiles_seq, feat, label = batch
            # 取第一个batch的第一个样本作为参考SMILES序列（固定）
            if ref_smiles_seq is None:
                ref_smiles_seq = smiles_seq[0:1].to(device)  # [1, 64]
            val_feat_list.extend(feat.numpy())
            val_label_list.extend(label.numpy())
    
    val_feat = np.array(val_feat_list)          # [N, 119]
    print(f"分子特征维度: {val_feat.shape}")
    print(f"参考SMILES序列维度: {ref_smiles_seq.shape}")
    
    # 2. 预测函数（接收numpy数组，返回概率）
    def predict_fn(feat_np):
        if feat_np.shape[0] == 0:
            return np.array([])
        model.eval()
        with torch.no_grad():
            feat_tensor = torch.FloatTensor(feat_np).to(device)
            # 检查是否有 NaN
            if torch.isnan(feat_tensor).any():
                print("Warning: NaN detected in feature tensor")
                feat_tensor = torch.nan_to_num(feat_tensor)
            batch_smiles = ref_smiles_seq.repeat(feat_np.shape[0], 1)
            logits = model(batch_smiles, feat_tensor)
            prob = torch.sigmoid(logits).cpu().numpy().flatten()
        return prob
    
    # 3. 设置SHAP解释器（KernelExplainer，GPU加速预测）
    # 背景数据：随机采样（根据数据量调整）
    background_size = min(100, len(val_feat))  
    background_feat = val_feat[np.random.choice(len(val_feat), background_size, replace=False)]
    print(f"背景数据维度: {background_feat.shape}")
    
    explainer = shap.KernelExplainer(predict_fn, background_feat)
    
    # 4. 计算SHAP值（增加样本量和nsamples）
    sample_size = min(50, len(val_feat))      
    sample_feat = val_feat[np.random.choice(len(val_feat), sample_size, replace=False)]
    nsamples = 500                                      # 增加采样次数（可调）
    print(f"分析样本维度: {sample_feat.shape}, nsamples = {nsamples}")
    
    shap_values = explainer.shap_values(sample_feat, nsamples=nsamples)
    
    # 5. 可视化（仅展示Top 20特征，避免图像过于拥挤）
    plt.rcParams["font.size"] = 10
    plt.rcParams["figure.figsize"] = (14, 10)
    
    # 5.1 特征重要性条形图（Top 20）
    shap.summary_plot(shap_values, sample_feat, feature_names=feat_cols,
                      plot_type="bar", max_display=20, show=False)
    plt.title("SHAP Feature Importance (Top 20)")
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_SAVE_DIR, "shap_bar_top20.png"), dpi=300, bbox_inches="tight")
    plt.close()
    
    # 5.2 特征贡献散点图（Top 20）
    shap.summary_plot(shap_values, sample_feat, feature_names=feat_cols,
                      max_display=20, show=False)
    plt.title("SHAP Feature Contribution (Top 20)")
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_SAVE_DIR, "shap_summary_top20.png"), dpi=300, bbox_inches="tight")
    plt.close()
    
    # 5.3 Top 5特征的依赖图（避免过多图像）
    shap_sum = np.abs(shap_values).mean(axis=0)
    top5_idx = np.argsort(shap_sum)[-5:]
    for idx in top5_idx:
        feat_name = feat_cols[idx]
        shap.dependence_plot(
            idx, shap_values, sample_feat, feature_names=feat_cols,
            show=False, x_jitter=0.1
        )
        plt.title(f"SHAP Dependence: {feat_name}")
        plt.tight_layout()
        plt.savefig(os.path.join(MODEL_SAVE_DIR, f"shap_dep_{feat_name}.png"), dpi=300, bbox_inches="tight")
        plt.close()
    
    print(f"\nSHAP分析完成！结果保存至 {MODEL_SAVE_DIR}")
    return shap_values

In [11]:
def plot_roc_curve(model, val_loader, save_path):
    """使用已训练模型和验证集绘制ROC曲线"""
    model.eval()
    y_true, y_pred_proba = [], []
    with torch.no_grad():
        for batch in val_loader:
            smiles_seq, feat, label = [x.to(DEVICE) for x in batch]
            logits = model(smiles_seq, feat)
            prob = torch.sigmoid(logits).cpu().numpy().flatten()
            y_pred_proba.extend(prob)
            y_true.extend(label.cpu().numpy().flatten())
    y_true = np.array(y_true)
    y_pred_proba = np.array(y_pred_proba)
    
    from sklearn.metrics import roc_curve, auc
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    # 最佳阈值（Youden指数）
    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[optimal_idx]
    
    plt.figure(figsize=(8,6))
    plt.plot(fpr, tpr, lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
    plt.plot([0,1],[0,1], 'k--', lw=1)
    plt.scatter(fpr[optimal_idx], tpr[optimal_idx], color='red', label=f'Optimal threshold = {optimal_threshold:.3f}')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ ROC曲线已保存至: {save_path}")

In [16]:
def aggregate_params(best_params_list):
    """将各折最优超参数聚合为最终参数（数值取平均，类别取众数，并确保 d_model 能被 nhead 整除）"""
    from collections import Counter
    
    aggregated = {}
    
    # 类别型参数：取众数
    categorical_params = ['nhead', 'batch_size']
    for key in categorical_params:
        values = [p[key] for p in best_params_list]
        aggregated[key] = Counter(values).most_common(1)[0][0]
    
    # 数值型参数：取算术平均
    numeric_params = ['d_model', 'num_layers', 'dropout', 'lr', 'weight_decay']
    for key in numeric_params:
        values = [p[key] for p in best_params_list]
        aggregated[key] = np.mean(values)
        # 对整数参数四舍五入
        if key in ['d_model', 'num_layers']:
            aggregated[key] = int(round(aggregated[key]))
    
    # 确保 d_model 能被 nhead 整除
    d_model = aggregated['d_model']
    nhead = aggregated['nhead']
    if d_model % nhead != 0:
        # 调整 d_model 到最近的能被 nhead 整除的数
        d_model_adjusted = (d_model // nhead) * nhead
        if d_model_adjusted == 0:
            d_model_adjusted = nhead  # 确保至少为 nhead
        print(f"调整 d_model: {d_model} -> {d_model_adjusted} 以保证能被 nhead={nhead} 整除")
        aggregated['d_model'] = d_model_adjusted
    
    return aggregated

In [17]:
# ===================== 7. 主程序入口 =====================
if __name__ == "__main__":
    set_seed(SEED)
    
    # 数据路径
    CSV_PATH = r"/root/molecular_descriptorsAML9.6-checkpoint.csv"
    
    # 1. 加载数据
    df, smiles_col, feat_cols, target_col = load_and_preprocess(CSV_PATH, target_col="Activity")
    print(f"数据加载完成，样本数: {len(df)}")
    
    # 2. 构建词汇表（全数据）
    vocab = build_vocab(df, smiles_col, min_freq=5)
    print(f"词汇表大小: {len(vocab)}")
    
    # 3. 嵌套交叉验证
    outer_scores, all_train_losses, all_val_losses, best_params_list = nested_cv(
        df, smiles_col, feat_cols, target_col, vocab, outer_k=5, inner_trials=6
    )
    
    # 4. 绘制并保存学习曲线
    plot_learning_curves(all_train_losses, all_val_losses, os.path.join(MODEL_SAVE_DIR, "learning_curves.png"))
    
    # 5. 保存各折性能指标到文件
    metrics_df = pd.DataFrame(outer_scores)
    metrics_df.to_csv(os.path.join(MODEL_SAVE_DIR, "outer_fold_metrics.csv"), index=False)
    print(f"✅ 各折指标已保存至 {MODEL_SAVE_DIR}/outer_fold_metrics.csv")

    # 6. 聚合各折最优超参数（取平均/众数）
    final_params = aggregate_params(best_params_list)
    print(f"聚合后的超参数: {final_params}")
    
    # 7. 在全部数据上训练最终模型
    final_model, final_scaler, final_val_loader = train_final_model(df, smiles_col, feat_cols, target_col, vocab, final_params, MODEL_SAVE_DIR)
    
    # 8. 最终模型验证集评估（含 F1 分数）
    final_model.eval()
    y_true_final, y_pred_final = [], []
    with torch.no_grad():
        for batch in final_val_loader:
            smiles_seq, feat, label = [x.to(DEVICE) for x in batch]
            logits = final_model(smiles_seq, feat)
            prob = torch.sigmoid(logits).cpu().numpy().flatten()
            y_pred_final.extend(prob)
            y_true_final.extend(label.cpu().numpy().flatten())
    
    final_metrics = calculate_metrics(np.array(y_true_final), np.array(y_pred_final), threshold=0.1)
    print("\n===== 最终模型在验证集上的性能 =====")
    for key, value in final_metrics.items():
        print(f"{key}: {value:.4f}")
    
    # 保存最终模型指标
    with open(os.path.join(MODEL_SAVE_DIR, "final_model_metrics.txt"), "w") as f:
        f.write("最终模型在验证集上的性能指标\n")
        f.write("=" * 40 + "\n")
        for key, value in final_metrics.items():
            f.write(f"{key}: {value:.4f}\n")
    print(f"✅ 最终模型指标已保存至 {MODEL_SAVE_DIR}/final_model_metrics.txt")
    
    # 9. SHAP 分析
    shap_values = shap_analysis(final_model, final_val_loader, final_scaler, feat_cols, device=DEVICE)
    
    # 10. ROC 曲线绘制
    plot_roc_curve(final_model, final_val_loader, os.path.join(MODEL_SAVE_DIR, "final_roc_curve.png"))

[I 2026-03-30 16:09:37,949] A new study created in memory with name: no-name-c9e1cecb-d136-4f01-9dc0-cb889220ee11


✅ 随机种子已设置为: 3407
数据加载完成，样本数: 511
词汇表大小: 33

========== Outer Fold 1/5 ==========
✅ 随机种子已设置为: 3407


[I 2026-03-30 16:09:41,741] Trial 0 finished with value: 0.8059752747252746 and parameters: {'d_model': 64, 'nhead': 4, 'num_layers': 2, 'dropout': 0.1853698155178157, 'lr': 0.00030268355570024763, 'weight_decay': 9.24235826795539e-05, 'batch_size': 16}. Best is trial 0 with value: 0.8059752747252746.


✅ 随机种子已设置为: 3407


[I 2026-03-30 16:09:42,351] Trial 1 finished with value: 0.7307692307692307 and parameters: {'d_model': 32, 'nhead': 4, 'num_layers': 1, 'dropout': 0.16956514659802213, 'lr': 0.00033392252599341387, 'weight_decay': 1.7103006284662822e-05, 'batch_size': 8}. Best is trial 0 with value: 0.8059752747252746.


...

✅ 随机种子已设置为: 3407


[I 2026-03-30 16:10:37,644] Trial 5 finished with value: 0.6600274725274725 and parameters: {'d_model': 32, 'nhead': 2, 'num_layers': 1, 'dropout': 0.17889876572231503, 'lr': 0.00019327623529487038, 'weight_decay': 9.143114520025115e-05, 'batch_size': 8}. Best is trial 0 with value: 0.8059752747252747.


Best params: {'d_model': 64, 'nhead': 4, 'num_layers': 2, 'dropout': 0.1853698155178157, 'lr': 0.00030268355570024763, 'weight_decay': 9.24235826795539e-05, 'batch_size': 16}
Best inner validation AUC: 0.8060
Fold 5 test metrics: Acc=0.8431, AUC=0.8914, Sen=0.7778, Spe=0.8667, F1=0.7241

===== Nested Cross-Validation Results =====
AUC: 0.8698 ± 0.0363
Accuracy: 0.8258 ± 0.0270
Sensitivity: 0.7373 ± 0.1028
Specificity: 0.8573 ± 0.0363
✅ 学习曲线保存至: chem_models_Transformer/learning_curves.png
✅ 各折指标已保存至 chem_models_Transformer/outer_fold_metrics.csv
调整 d_model: 58 -> 56 以保证能被 nhead=4 整除
聚合后的超参数: {'nhead': 4, 'batch_size': 16, 'd_model': 56, 'num_layers': 2, 'dropout': np.float64(0.18220888173385702), 'lr': np.float64(0.00030893134975888085), 'weight_decay': np.float64(7.73594674005757e-05)}
✅ 最终模型已保存至: /root/final_transformer_model.pth

===== 最终模型在验证集上的性能 =====
accuracy: 0.8350
sensitivity: 0.7037
specificity: 0.8816
precision: 0.6786
f1: 0.6909
auc: 0.8635
✅ 最终模型指标已保存至 chem_models_Transfor

100%|██████████| 50/50 [00:22<00:00,  2.19it/s]



SHAP分析完成！结果保存至 chem_models_Transformer
✅ ROC曲线已保存至: chem_models_Transformer/final_roc_curve.png
